In [28]:
# Generate six simulated vegetation abundance datasets per user's scenarios
import numpy as np
import pandas as pd
import zipfile
from pathlib import Path

In [29]:
rng = np.random.default_rng(42)

In [30]:
PLOTS = 250
SCENARIOS = [
    ("Sep0",   0.0,  140),
    ("Sep1",  -0.1,  105),
    ("Sep2",  -0.2,  178),
    ("Sep3",  -0.3,  167),
    ("Sep4",  -0.4,   98),
    ("Sep5",  -0.5,  118),
]

# Hyperparameters
G = 3  # latent habitats/communities
eta = 0.2  # controls sparsity / distinctness of habitat compositions (smaller => sparser)
base_kappa = 0.05  # near one-hot (strong separation)
kappa_slope = 4.0  # how fast mixing increases with |sep|
mean_count = 120   # mean total counts per plot
dispersion = 0.6   # for NegBin (smaller => more overdispersion)

out_dir = Path("/mnt/data/veg_sims")
out_dir.mkdir(parents=True, exist_ok=True)
meta_rows = []

def negbin_counts(mu, phi, rng):
    """
    Draw counts N ~ NegBin with mean mu and 'size' phi using Poisson-Gamma mixture.
    Variance = mu + mu^2/phi
    """
    # Gamma shape=phi, scale=mu/phi
    lam = rng.gamma(shape=phi, scale=mu/phi)
    return rng.poisson(lam)

def simulate_scenario(tag, sep_value, n_species):
    # Species names
    species = [f"sp_{i+1}" for i in range(n_species)]
    
    # Habitat-specific species compositions (phi_g)
    # Draw G distinct Dirichlet distributions; smaller eta -> peakier, more distinct profiles
    phi = rng.dirichlet(alpha=np.full(n_species, eta), size=G)  # (G, S)
    
    # Mixing concentration (Dirichlet kappa) controls plot-level blend among habitats
    kappa = base_kappa + kappa_slope * abs(sep_value)
    kappavec = np.full(G, kappa)
    
    # Simulate plots
    abundance = np.zeros((PLOTS, n_species), dtype=int)
    mix_weights = np.zeros((PLOTS, G))
    totals = np.zeros(PLOTS, dtype=int)
    for p in range(PLOTS):
        w = rng.dirichlet(kappavec)  # habitat mixture for this plot
        mix_weights[p] = w
        theta = (w @ phi)  # resulting species proportions (S,)
        theta = theta / theta.sum()  # guard (should already sum ~1)
        
        # total count per plot from NegBin (overdispersed), then split via multinomial
        N = int(negbin_counts(mean_count, 1.0/dispersion, rng))
        # ensure at least some counts:
        N = max(N, 1)
        totals[p] = N
        abundance[p] = rng.multinomial(N, theta)
    
    # DataFrames
    df_abund = pd.DataFrame(abundance, columns=species)
    df_abund.index = [f"plot_{i+1}" for i in range(PLOTS)]
    
    df_mix = pd.DataFrame(mix_weights, columns=[f"hab_{g+1}" for g in range(G)])
    df_mix.index = df_abund.index
    df_mix["total_count"] = totals
    
    # Save
    abund_path = out_dir / f"{tag}_abundance.csv"
    mix_path = out_dir / f"{tag}_mixweights.csv"
    phi_path = out_dir / f"{tag}_habitat_profiles.csv"
    
    df_abund.to_csv(abund_path, index=True)
    df_mix.to_csv(mix_path, index=True)
    pd.DataFrame(phi, columns=species, index=[f"hab_{g+1}" for g in range(G)]).to_csv(phi_path, index=True)
    
    meta_rows.append({
        "scenario": tag,
        "separation_value": sep_value,
        "species": n_species,
        "plots": PLOTS,
        "G_habitats": G,
        "kappa_used": kappa,
        "abundance_csv": str(abund_path),
        "mixweights_csv": str(mix_path),
        "habitat_profiles_csv": str(phi_path),
        "mean_total_counts_approx": df_mix["total_count"].mean()
    })
    
    return df_abund.head(8), df_mix.head(8)

# Run all scenarios and preview
previews = []
for tag, sep, S in SCENARIOS:
    prev_abund, prev_mix = simulate_scenario(tag, sep, S)
    previews.append((tag, prev_abund, prev_mix))


In [31]:
import pandas as pd
import zipfile
import os
from pathlib import Path

# 1. Kayıt yollarını belirleyelim (Masaüstü hedeflendi)
desktop_path = Path(r"C:\Users\nursu\OneDrive\Masaüstü")
zip_path = desktop_path / "simulated_data.zip"
meta_path = desktop_path / "metadata.csv"

# 2. Metadata CSV dosyasını oluştur ve Masaüstüne kaydet
df_meta = pd.DataFrame(meta_rows)
df_meta.to_csv(meta_path, index=False)

# 3. ZIP paketini oluştur
# Bu blok, hem metadata'yı hem de meta_rows içindeki tüm CSV'leri zipe ekler
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    # Metadata dosyasını ekle
    zf.write(meta_path, arcname="metadata.csv")
    
    # meta_rows içindeki her bir simülasyon dosyasını ekle
    for row in meta_rows:
        zf.write(row["abundance_csv"], arcname=Path(row["abundance_csv"]).name)
        zf.write(row["mixweights_csv"], arcname=Path(row["mixweights_csv"]).name)
        zf.write(row["habitat_profiles_csv"], arcname=Path(row["habitat_profiles_csv"]).name)

# 4. Önizleme (caas_jupyter_tools hatası almamak için standart display kullanıyoruz)
print(f"✅ İşlem başarıyla tamamlandı!")
print(f"📍 ZIP dosyası burada: {zip_path}")
print(f"📍 Meta dosyası burada: {meta_path}")

# Jupyter üzerinde veriyi görmek için:
display(df_meta.head())

✅ İşlem başarıyla tamamlandı!
📍 ZIP dosyası burada: C:\Users\nursu\OneDrive\Masaüstü\simulated_data.zip
📍 Meta dosyası burada: C:\Users\nursu\OneDrive\Masaüstü\metadata.csv


,scenario,separation_value,species,plots,G_habitats,kappa_used,abundance_csv,mixweights_csv,habitat_profiles_csv,mean_total_counts_approx
0,Sep0,0.0,140,250,3,0.05,\mnt\data\veg_sims\Sep0_abundance.csv,\mnt\data\veg_sims\Sep0_mixweights.csv,\mnt\data\veg_sims\Sep0_habitat_profiles.csv,116.688
1,Sep1,-0.1,105,250,3,0.45,\mnt\data\veg_sims\Sep1_abundance.csv,\mnt\data\veg_sims\Sep1_mixweights.csv,\mnt\data\veg_sims\Sep1_habitat_profiles.csv,122.200
2,Sep2,-0.2,178,250,3,0.85,\mnt\data\veg_sims\Sep2_abundance.csv,\mnt\data\veg_sims\Sep2_mixweights.csv,\mnt\data\veg_sims\Sep2_habitat_profiles.csv,118.516
3,Sep3,-0.3,167,250,3,1.25,\mnt\data\veg_sims\Sep3_abundance.csv,\mnt\data\veg_sims\Sep3_mixweights.csv,\mnt\data\veg_sims\Sep3_habitat_profiles.csv,128.408
4,Sep4,-0.4,98,250,3,1.65,\mnt\data\veg_sims\Sep4_abundance.csv,\mnt\data\veg_sims\Sep4_mixweights.csv,\mnt\data\veg_sims\Sep4_habitat_profiles.csv,125.092
